# Accuracy tables: Uniform K, Blind, Ours, GazeVLM, StreamingTOM, DyCoke, FastVID, PruneVid, DTD, Uniform K + average pooling and the two-view GazeVLM on six VLMs (Colab)

Runs `accuracy_tables/` (see its README.md) on a Colab **A100** runtime with the project data on Google Drive
(`MyDrive/GazeVLM/`). Elsewhere, run the same commands from a shell with your own `config.yaml`.

1. Clone and install  2. Drive, Hugging Face login, config  3. Unit tests  4. Score  5. Summary

Results are saved after every question; rerun a cell to resume.

In [ ]:
import os, sys, subprocess
import torch

# An A100 (bf16, 40 GB+) is needed: on a T4 (16 GB, no bf16) Uniform K's 96 frames run out of memory and
# the bf16 models crawl. Runtime > Change runtime type > A100 GPU.
gpu = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
print('GPU:', f'{gpu.name}, {gpu.total_memory / 1e9:.0f} GB' if gpu else 'none')
assert gpu and torch.cuda.get_device_capability(0)[0] >= 8 and gpu.total_memory > 35e9, \
    'needs an A100 GPU runtime (bf16, >= 40 GB): Runtime > Change runtime type > A100'

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
# num2words: needed by SmolVLM2's processor (install before transformers is first imported)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pytest', 'num2words', 'pyyaml'],
               check=True)

In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')

try:                                         # Hugging Face token from the Colab secret HF_TOKEN (Gemma is gated)
    from google.colab import userdata
    from huggingface_hub import login
    if userdata.get('HF_TOKEN'):
        login(token=userdata.get('HF_TOKEN'))
except Exception as ex:
    print('no HF_TOKEN secret:', type(ex).__name__)

CONFIG = 'accuracy_tables/config.yaml'
# The example config already points at MyDrive/GazeVLM; edit CONFIG after this cell to change paths or options.
if not os.path.exists(CONFIG):
    shutil.copy('accuracy_tables/config.example.yaml', CONFIG)
print(open(CONFIG).read())

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_accuracy_tables.py', 'tests/test_gazevlm.py', 'tests/test_streamingtom.py', 'tests/test_dycoke.py', 'tests/test_fastvid.py', 'tests/test_prunevid.py', 'tests/test_dtd.py', 'tests/test_avgpool.py', 'tests/test_gazevlm2.py', '-q'],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, 'unit tests failed'

## Score

`SMOKE = True` (default): a quick check that every method runs on every model: **one video, one question**,
all rows on all six VLMs in one process, results in `/content/smoke` (wiped first, never mixed with the real
results), a failing method does not stop the others, and a **pass / fail table** per model and row is printed at
the end. About a few minutes per model (Uniform K on Phi-3.5, 96 frames x 757 tokens, is the slowest row).

`SMOKE = False`: the real run. `MODELS`: any of qwen2.5, qwen3, internvl, gemma, phi, smolvlm. `ROWS`: uniform, blind,
ours, gazevlm, streamingtom, dycoke, fastvid, prunevid, dtd, avgpool, gazevlm2. `VIDEOS`: test (4 locked test videos,
204 questions) or all (12 videos, 834 questions).
Each model runs its parity check first and is skipped (with the reason printed) if it fails.

In [ ]:
MODELS = ['qwen2.5', 'qwen3', 'internvl', 'gemma', 'phi', 'smolvlm']
ROWS = 'uniform,blind,ours,gazevlm,streamingtom,dycoke,fastvid,prunevid,dtd,avgpool,gazevlm2'
SMOKE = True                   # True: 1 video x 1 question, every row on every model, pass / fail table; False: real run

if SMOKE:
    SMOKE_DIR = '/content/smoke'
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)          # always a fresh check
    get_ipython().system(f'python -m accuracy_tables.run --config {CONFIG} --model {",".join(MODELS)} --rows {ROWS} '
                         f'--videos loc2_script1_seq2_rec1 --max-questions 1 --out-dir {SMOKE_DIR} --keep-going')
else:
    VIDEOS = 'test'            # the 4 locked test videos (204 questions); 'all' = 12 videos
    for m in MODELS:
        get_ipython().system(f'python -m accuracy_tables.run --config {CONFIG} --model {m} --rows {ROWS} --videos {VIDEOS}')

## Summary

Accuracy with a 95% interval over videos, visual tokens per question and the token budget vs
each model's Uniform K row. Also written to `<out_dir>/summary.csv`.

In [ ]:
if SMOKE:
    print('smoke test: see the pass / fail table above (per-row files in /content/smoke/<model>/)')
else:
    get_ipython().system(f'python -m accuracy_tables.summarize --config {CONFIG} --videos {VIDEOS}')